# 10 - Dataset audit (Phase A)

Second NCBI retrieval for the **annotation-enriched candidate set**, plus the
audit artifacts that go with it:

- ESearch **preflight** (`Count`, `translated_query`, sample QC, `max_uid_count` guard)
- UID -> XML -> flattened metadata snapshots (reusing the existing NCBI modules)
- `query_reference_recall` (overall + per MID-PIWI clade)
- `pago_technical_prefilter` (technical exclusions only)
- `derived_protein_fasta` for the `annotation_enriched_proteome` dataset

The candidate set is called *annotation-enriched* on purpose: a text query for
`PIWI` / `Argonaute` recovers proteins already annotated with that terminology,
not the full universe of pAgos. `query_reference_recall` measures how much of a
curated reference set it misses; a sequence-based discovery route is future work.

Notebooks are orchestration only. Logic lives in `src/pago_pipeline/`.

In [ ]:
# =============================================================================
# CELL 1 - Imports
# =============================================================================

from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.ncbi_esearch_preflight_snapshot as esearch_preflight_snapshot_module
import src.pago_pipeline.ncbi_metadata_snapshot as ncbi_metadata_snapshot_module
import src.pago_pipeline.query_reference_recall_snapshot as query_reference_recall_snapshot_module
import src.pago_pipeline.pago_technical_prefilter_snapshot as pago_technical_prefilter_snapshot_module
import src.pago_pipeline.derived_protein_fasta_snapshot as derived_protein_fasta_snapshot_module

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
esearch_preflight_snapshot_module = importlib.reload(esearch_preflight_snapshot_module)
ncbi_metadata_snapshot_module = importlib.reload(ncbi_metadata_snapshot_module)
query_reference_recall_snapshot_module = importlib.reload(query_reference_recall_snapshot_module)
pago_technical_prefilter_snapshot_module = importlib.reload(pago_technical_prefilter_snapshot_module)
derived_protein_fasta_snapshot_module = importlib.reload(derived_protein_fasta_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
resolve_ncbi_protein_uid_snapshot = ncbi_snapshot_module.resolve_ncbi_protein_uid_snapshot
resolve_ncbi_protein_xml_snapshot = ncbi_snapshot_module.resolve_ncbi_protein_xml_snapshot
resolve_ncbi_protein_metadata_snapshot = (
    ncbi_metadata_snapshot_module.resolve_ncbi_protein_metadata_snapshot
)
resolve_ncbi_esearch_preflight_snapshot = (
    esearch_preflight_snapshot_module.resolve_ncbi_esearch_preflight_snapshot
)
resolve_query_reference_recall_snapshot = (
    query_reference_recall_snapshot_module.resolve_query_reference_recall_snapshot
)
resolve_pago_technical_prefilter_snapshot = (
    pago_technical_prefilter_snapshot_module.resolve_pago_technical_prefilter_snapshot
)
resolve_derived_protein_fasta_snapshot = (
    derived_protein_fasta_snapshot_module.resolve_derived_protein_fasta_snapshot
)
DEFAULT_RETAINED_PROTEIN_UIDS_FILE_NAME = (
    pago_technical_prefilter_snapshot_module.DEFAULT_RETAINED_PROTEIN_UIDS_FILE_NAME
)
PAGO_TECHNICAL_PREFILTER_ARTIFACT_TYPE = pago_technical_prefilter_snapshot_module.ARTIFACT_TYPE

import pandas as pd
from src.pago_pipeline.pago_technical_prefilter import build_technical_prefilter_policy_sha256


In [ ]:
# =============================================================================
# CELL 2 - Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)
if not dotenv_path:
    raise FileNotFoundError(
        'Could not find a .env file while walking up parent directories. '
        'Place .env with your NCBI email and optional API key at the project root.'
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

NCBI_EMAIL = os.getenv('NCBI_EMAIL')
NCBI_API_KEY = os.getenv('NCBI_API_KEY')
if not NCBI_EMAIL:
    raise ValueError('NCBI_EMAIL was not found in the environment. Define it in your .env file.')

print(f'Project root: {PROJECT_ROOT}')
print(f'NCBI email configured: {bool(NCBI_EMAIL)}')
print(f'NCBI API key configured: {bool(NCBI_API_KEY)}')


In [ ]:
# =============================================================================
# CELL 3 - Configuration
# =============================================================================

DATASET_NAME = 'annotation_enriched_candidate_set'
SEARCH_QUERY = (
    '(PIWI[All Fields] OR Argonaute[All Fields]) '
    'AND (Bacteria[Organism] OR Archaea[Organism])'
)

# Preflight guard. If NCBI reports more than MAX_UID_COUNT protein UIDs, the
# preflight report is still materialized but the full retrieval is blocked
# until you review it and set ALLOW_EXCEEDS_MAX_UID_COUNT = True deliberately.
MAX_UID_COUNT = 250_000
PREFLIGHT_SAMPLE_SIZE = 200
ALLOW_EXCEEDS_MAX_UID_COUNT = False

# UID retrieval (mirrors notebook 00).
UID_PAGE_SIZE = 10_000
UID_MAX_RETRY_ATTEMPTS = 5
UID_REQUEST_DELAY_SECONDS = None
UID_FETCH_TIMEOUT_SECONDS = 30.0
UID_REQUEST_DEADLINE_SECONDS = 300.0

# XML retrieval (mirrors notebook 01).
XML_BATCH_SIZE = 100
XML_MAX_RETRY_ATTEMPTS = 5
XML_MAX_CONCURRENT_REQUESTS = 4
XML_REUSE_HTTP_CONNECTION = False
XML_ENABLE_BATCH_RESUME = True
XML_PURGE_BATCH_WORKSPACE_ON_SUCCESS = True

SEQUENCE_LINE_WIDTH = 60
UPDATE_LATEST_DIRECTORY = True

# Snapshot modes. reuse_latest_or_create keeps a rerun cheap.
PREFLIGHT_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
UID_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
XML_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
METADATA_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
QUERY_REFERENCE_RECALL_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
TECHNICAL_PREFILTER_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create
DERIVED_FASTA_SNAPSHOT_MODE = SnapshotMode.reuse_latest_or_create

print(f'Dataset name:  {DATASET_NAME}')
print(f'Search query:  {SEARCH_QUERY}')
print(f'max_uid_count: {MAX_UID_COUNT}')


In [ ]:
# =============================================================================
# CELL 4 - Paths
# =============================================================================

DATA_ROOT = PROJECT_ROOT / 'data'

PREFLIGHT_SNAPSHOT_ROOT_DIRECTORY = (
    DATA_ROOT / '01-raw' / f'esearch_preflight__{DATASET_NAME}'
)
UID_SNAPSHOT_ROOT_DIRECTORY = (
    DATA_ROOT / '01-raw' / f'protein_uid_snapshots__{DATASET_NAME}'
)
XML_SNAPSHOT_ROOT_DIRECTORY = (
    DATA_ROOT / '01-raw' / f'protein_xml_snapshots__{DATASET_NAME}'
)
METADATA_SNAPSHOT_ROOT_DIRECTORY = (
    DATA_ROOT / '02-intermediate' / f'protein_metadata_csv__{DATASET_NAME}'
)
QUERY_REFERENCE_RECALL_SNAPSHOT_ROOT_DIRECTORY = (
    DATA_ROOT / '02-intermediate' / f'query_reference_recall__{DATASET_NAME}'
)
TECHNICAL_PREFILTER_SNAPSHOT_ROOT_DIRECTORY = (
    DATA_ROOT / '03-features' / 'pago_technical_prefilter'
)
DERIVED_FASTA_SNAPSHOT_ROOT_DIRECTORY = (
    DATA_ROOT / '02-intermediate' / 'derived_protein_fasta__annotation_enriched_proteome'
)

QUERY_RECALL_REFERENCE_SET_CSV_PATH = (
    PROJECT_ROOT / 'tests' / 'fixtures' / 'query_recall_reference_set.csv'
)

for path in (
    PREFLIGHT_SNAPSHOT_ROOT_DIRECTORY,
    UID_SNAPSHOT_ROOT_DIRECTORY,
    XML_SNAPSHOT_ROOT_DIRECTORY,
    METADATA_SNAPSHOT_ROOT_DIRECTORY,
    QUERY_REFERENCE_RECALL_SNAPSHOT_ROOT_DIRECTORY,
    TECHNICAL_PREFILTER_SNAPSHOT_ROOT_DIRECTORY,
    DERIVED_FASTA_SNAPSHOT_ROOT_DIRECTORY,
):
    print(path)


In [ ]:
# =============================================================================
# CELL 5 - ESearch preflight
# =============================================================================

preflight_payload = resolve_ncbi_esearch_preflight_snapshot(
    snapshot_mode=PREFLIGHT_SNAPSHOT_MODE,
    snapshot_root_directory=PREFLIGHT_SNAPSHOT_ROOT_DIRECTORY,
    search_query=SEARCH_QUERY,
    ncbi_email=NCBI_EMAIL,
    ncbi_api_key=NCBI_API_KEY,
    max_uid_count=MAX_UID_COUNT,
    sample_size=PREFLIGHT_SAMPLE_SIZE,
    allow_exceeds_max_uid_count=ALLOW_EXCEEDS_MAX_UID_COUNT,
)
preflight_manifest = preflight_payload['manifest']

print(f"NCBI Count:            {preflight_manifest['result_count']:,}")
print(f"translated_query:      {preflight_manifest['translated_query']}")
print(f"exceeds max_uid_count: {preflight_manifest['exceeds_max_uid_count']}")
print(f"sample records:        {preflight_manifest['sample_record_count']}")
print(f"  with sequence:       {preflight_manifest['sample_records_with_sequence']}")
print(f"  missing sequence:    {preflight_manifest['sample_records_missing_sequence']}")
print(f"  extractable uid:     {preflight_manifest['sample_records_with_extractable_uid']}")
print(f"sample fetch error:   {preflight_manifest['sample_fetch_error']}")


In [ ]:
# =============================================================================
# CELL 6 - Protein UID snapshot
# =============================================================================

uid_snapshot_payload = resolve_ncbi_protein_uid_snapshot(
    snapshot_mode=UID_SNAPSHOT_MODE,
    snapshot_root_directory=UID_SNAPSHOT_ROOT_DIRECTORY,
    search_query=SEARCH_QUERY,
    page_size=UID_PAGE_SIZE,
    max_retry_attempts=UID_MAX_RETRY_ATTEMPTS,
    request_delay_seconds=UID_REQUEST_DELAY_SECONDS,
    fetch_timeout_seconds=UID_FETCH_TIMEOUT_SECONDS,
    request_deadline_seconds=UID_REQUEST_DEADLINE_SECONDS,
    ncbi_email=NCBI_EMAIL,
    ncbi_api_key=NCBI_API_KEY,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)
active_protein_uid_list = uid_snapshot_payload['protein_uids']
print(f'Protein UIDs retrieved: {len(active_protein_uid_list):,}')
print(f"UID snapshot dir:       {uid_snapshot_payload['snapshot_directory']}")


In [ ]:
# =============================================================================
# CELL 7 - Protein XML snapshot
# =============================================================================

xml_snapshot_payload = resolve_ncbi_protein_xml_snapshot(
    snapshot_mode=XML_SNAPSHOT_MODE,
    snapshot_root_directory=XML_SNAPSHOT_ROOT_DIRECTORY,
    source_uid_snapshot_root_directory=UID_SNAPSHOT_ROOT_DIRECTORY,
    xml_batch_size=XML_BATCH_SIZE,
    max_retry_attempts=XML_MAX_RETRY_ATTEMPTS,
    max_concurrent_requests=XML_MAX_CONCURRENT_REQUESTS,
    reuse_http_connection=XML_REUSE_HTTP_CONNECTION,
    enable_batch_resume=XML_ENABLE_BATCH_RESUME,
    purge_batch_workspace_on_success=XML_PURGE_BATCH_WORKSPACE_ON_SUCCESS,
    ncbi_email=NCBI_EMAIL,
    ncbi_api_key=NCBI_API_KEY,
    update_latest_directory=UPDATE_LATEST_DIRECTORY,
)
xml_manifest = xml_snapshot_payload['manifest']
print(f"XML records: {xml_manifest['consolidated_record_count']:,}")
print(f"XML snapshot dir: {xml_snapshot_payload['snapshot_directory']}")


In [ ]:
# =============================================================================
# CELL 8 - Flattened metadata snapshot
# =============================================================================

metadata_snapshot_payload = resolve_ncbi_protein_metadata_snapshot(
    snapshot_mode=METADATA_SNAPSHOT_MODE,
    snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    source_xml_snapshot_root_directory=XML_SNAPSHOT_ROOT_DIRECTORY,
)
metadata_manifest = metadata_snapshot_payload['manifest']
METADATA_CSV_FILE_PATH = metadata_snapshot_payload['csv_file_path']
print(f"Metadata rows:    {metadata_manifest['row_count']:,}")
print(f"Metadata columns: {metadata_manifest['column_count']}")
print(f"Metadata CSV:     {METADATA_CSV_FILE_PATH}")


In [ ]:
# =============================================================================
# CELL 9 - Query reference recall (annotation-enriched set coverage)
# =============================================================================

recall_payload = resolve_query_reference_recall_snapshot(
    snapshot_mode=QUERY_REFERENCE_RECALL_SNAPSHOT_MODE,
    snapshot_root_directory=QUERY_REFERENCE_RECALL_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    query_recall_reference_set_csv_path=QUERY_RECALL_REFERENCE_SET_CSV_PATH,
)
recall_manifest = recall_payload['manifest']
stratum_recall = recall_manifest['stratum_recall']
stratum_recall_status = recall_manifest['stratum_recall_status']

def _fmt_recall(metric_name):
    value = stratum_recall.get(metric_name)
    status = stratum_recall_status.get(metric_name)
    if value is None:
        return f'NOT_EVALUABLE ({status})'
    return f'{value:.3f}'

print(f"reference pAgos:  {recall_manifest['reference_count']}")
print(f"recovered:        {recall_manifest['recovered_count']}")
for metric_name in (
    'overall_reference_recall',
    'long_a_reference_recall',
    'long_b_reference_recall',
    'short_reference_recall',
    'piwi_re_reference_recall',
):
    print(f'{metric_name}: {_fmt_recall(metric_name)}')
display(recall_payload['summary'])
display(recall_payload['detail'])


In [ ]:
# =============================================================================
# CELL 10 - Technical prefilter
# =============================================================================

prefilter_payload = resolve_pago_technical_prefilter_snapshot(
    snapshot_mode=TECHNICAL_PREFILTER_SNAPSHOT_MODE,
    snapshot_root_directory=TECHNICAL_PREFILTER_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
)
prefilter_manifest = prefilter_payload['manifest']

print(f"input records:    {prefilter_manifest['input_record_count']:,}")
print(f"retained:         {prefilter_manifest['retained_record_count']:,}")
print(f"excluded (tech.): {prefilter_manifest['excluded_record_count']:,}")
print(f"policy sha256:    {prefilter_manifest['technical_prefilter_policy_sha256']}")
assert prefilter_manifest['technical_prefilter_policy_sha256'] == build_technical_prefilter_policy_sha256()
print()
print('technical exclusions by reason (retain is not an exclusion):')
for decision, count in prefilter_manifest['counts_by_decision'].items():
    if decision == 'retain':
        continue
    print(f'  {decision}: {count:,}')
display(prefilter_payload['prefilter_counts'])


In [ ]:
# =============================================================================
# CELL 11 - Derived FASTA (annotation_enriched_proteome)
# =============================================================================

derived_fasta_payload = resolve_derived_protein_fasta_snapshot(
    snapshot_mode=DERIVED_FASTA_SNAPSHOT_MODE,
    snapshot_root_directory=DERIVED_FASTA_SNAPSHOT_ROOT_DIRECTORY,
    source_metadata_snapshot_root_directory=METADATA_SNAPSHOT_ROOT_DIRECTORY,
    source_selection_snapshot_root_directory=TECHNICAL_PREFILTER_SNAPSHOT_ROOT_DIRECTORY,
    selection_artifact_type=PAGO_TECHNICAL_PREFILTER_ARTIFACT_TYPE,
    selection_uid_list_file_name=DEFAULT_RETAINED_PROTEIN_UIDS_FILE_NAME,
    record_selection_rule='pago_technical_prefilter.retained',
    record_selection_config_sha256=prefilter_manifest['technical_prefilter_policy_sha256'],
    dataset_kind='annotation_enriched_proteome',
    sequence_line_width=SEQUENCE_LINE_WIDTH,
    record_order='as_selected',
)
derived_fasta_manifest = derived_fasta_payload['manifest']

print(f"artifact_type:        {derived_fasta_manifest['artifact_type']}")
print(f"dataset_kind:         {derived_fasta_manifest['dataset_kind']}")
print(f"requested uids:       {derived_fasta_manifest['requested_uid_count']:,}")
print(f"resolved uids:        {derived_fasta_manifest['resolved_uid_count']:,}")
print(f"FASTA records:        {derived_fasta_manifest['fasta_record_count']:,}")
print(f"record_order:         {derived_fasta_manifest['record_order']}")
print(f"source_record_ids_sha256: {derived_fasta_manifest['source_record_ids_sha256']}")
print(f"fasta_file_sha256:        {derived_fasta_manifest['fasta_file_sha256']}")


In [ ]:
# =============================================================================
# CELL 12 - Audit summary
# =============================================================================

audit_summary = {
    'dataset_name': DATASET_NAME,
    'search_query': SEARCH_QUERY,
    'translated_query': preflight_manifest['translated_query'],
    'ncbi_count': preflight_manifest['result_count'],
    'uids_retrieved': len(active_protein_uid_list),
    'xml_records': xml_manifest['consolidated_record_count'],
    'metadata_rows': metadata_manifest['row_count'],
    'prefilter_input': prefilter_manifest['input_record_count'],
    'prefilter_retained': prefilter_manifest['retained_record_count'],
    'prefilter_excluded': prefilter_manifest['excluded_record_count'],
    'prefilter_excluded_by_reason': prefilter_manifest['counts_by_decision'],
    'overall_reference_recall': stratum_recall['overall_reference_recall'],
    'long_a_reference_recall': stratum_recall['long_a_reference_recall'],
    'long_b_reference_recall': stratum_recall['long_b_reference_recall'],
    'short_reference_recall': stratum_recall['short_reference_recall'],
    'piwi_re_reference_recall': stratum_recall['piwi_re_reference_recall'],
    'derived_fasta_records': derived_fasta_manifest['fasta_record_count'],
}
for key, value in audit_summary.items():
    print(f'{key}: {value}')

# Length distribution of retained records (length_warning is informational only).
retained_records = prefilter_payload['retained_records']
if 'gbseq__length' in retained_records.columns:
    lengths = pd.to_numeric(retained_records['gbseq__length'], errors='coerce').dropna()
    print()
    print('retained sequence length distribution:')
    print(lengths.describe())
    print(f"length_warning=True: {int(retained_records['length_warning'].sum()):,}")


In [ ]:
# =============================================================================
# CELL 13 - Expose downstream variables
# =============================================================================

print('Variables exposed for downstream notebooks:')
print('- DATASET_NAME, SEARCH_QUERY')
print('- METADATA_SNAPSHOT_ROOT_DIRECTORY, METADATA_CSV_FILE_PATH')
print('- TECHNICAL_PREFILTER_SNAPSHOT_ROOT_DIRECTORY')
print('- DERIVED_FASTA_SNAPSHOT_ROOT_DIRECTORY')
print('- preflight_payload, uid_snapshot_payload, xml_snapshot_payload')
print('- metadata_snapshot_payload, recall_payload, prefilter_payload, derived_fasta_payload')
print('- audit_summary')
